<a href="https://colab.research.google.com/github/AlvaroAla/TE-IA/blob/main/GSI073_aula0_support_vector_machine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GSI073 - Tópicos Especiais de Inteligência Artificial

Neste notebook, um tipo de Support Vector Machine Linear.


## Preparação dos dados

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn import datasets

# Preparar o dataset
iris = datasets.load_iris()
X = iris.data; y = iris.target

X = X[y != 1] ; y = y[y != 1] # versicolor
y = torch.tensor(y, dtype=torch.float32)
y[y == 0] = -1  # SVM espera rótulos em {-1, +1}

X = torch.tensor(X, dtype=torch.float32) # Tensor é um tipo especial que suporta muitas dimensões

A nossa Support Vector Machine é basicamente um hiperplano definido por *w* e *b* que melhor separa as classes.

In [ ]:
# Definir parâmetros treináveis da Support Vector Machine: w e b
n_features = X.shape[1]
w = torch.randn(n_features, 1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

# === Hiperparâmetros ===
learning_rate = 0.01
epochs = 300
optimizer = optim.Adam([w, b], lr=learning_rate)

## Execução do treinamento

In [ ]:
for epoch in range(epochs):
    optimizer.zero_grad()

    y_pred = X @ w + b  # Modelo SVM (um hiperplano que depende de w e b)

    # Hinge loss: max(0, 1 - y_i * (w^T x_i + b))
    perda_de_classificacao = torch.clamp(1 - y.view(-1, 1) * y_pred, min=0).mean()

    # Termo de regularização
    perda_de_distancia_entre_classes = 0.5 * torch.sum(w ** 2) # 2/||w|| é a distância que queremos que seja a maior possível

    # Função objetivo tradicional: minimizar reg + C * hinge
    loss = perda_de_distancia_entre_classes + perda_de_classificacao

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss={loss.item():.4f}")

##Resposta do exercicio

In [ ]:
# === Avaliação do modelo SVM Treinado ===

from sklearn.metrics import accuracy_score, confusion_matrix

with torch.no_grad():
    y_scores = (X @ w + b).view(-1)
    y_pred_svm = torch.sign(y_scores)         # converte logits → {-1, +1}

# converter para numpy
y_true_np = y.numpy()
y_pred_np = y_pred_svm.numpy()

acc_svm = accuracy_score(y_true_np, y_pred_np)
cm_svm = confusion_matrix(y_true_np, y_pred_np)

print("Acurácia da SVM:", acc_svm)
print("\nMatriz de Confusão:\n", cm_svm)

# Mostrar algumas predições
for i in range(5):
    print(f"Amostra {i}: Score={y_scores[i].item():.4f}, Pred={y_pred_np[i]}, Real={y_true_np[i]}")


In [ ]:
# === Comparação com Regressão Logística (versão DEFINITIVA) ===

# rótulos 0/1, convertidos explicitamente para numpy inteiro
y_log = ((y + 1) / 2).view(-1).detach().cpu().numpy().astype("int64")

modelo_log = nn.Linear(4, 1)
criterion = nn.BCEWithLogitsLoss()
optimizer_log = optim.Adam(modelo_log.parameters(), lr=0.05)

# Treinar
for epoch in range(300):
    optimizer_log.zero_grad()
    logits = modelo_log(X).view(-1)
    loss_log = criterion(logits, torch.tensor(y_log, dtype=torch.float32))
    loss_log.backward()
    optimizer_log.step()

# Predição
with torch.no_grad():
    probs = torch.sigmoid(modelo_log(X).view(-1)).detach().cpu().numpy()
    y_pred_log = (probs >= 0.5).astype("int64")

# Agora SÓ existem 0/1 de cada lado
acc_log = accuracy_score(y_log, y_pred_log)

print("Acurácia SVM:", acc_svm)
print("Acurácia Regressão Logística:", acc_log)


In [ ]:
# === Criando nn.Linear equivalente a w e b ===

modelo_equivalente = nn.Linear(n_features, 1)

# Copiar pesos da SVM treinada
with torch.no_grad():
    modelo_equivalente.weight.copy_(w.view(1, -1))   # w^T
    modelo_equivalente.bias.copy_(b)

print("=== Pesos originais w e b ===")
print("w:", w.view(1, -1))
print("b:", b)

print("\n=== Pesos dentro de nn.Linear ===")
print(modelo_equivalente.weight)
print(modelo_equivalente.bias)

# Testar se produzem a mesma saída
with torch.no_grad():
    original_scores = (X @ w + b).view(-1)
    linear_scores = modelo_equivalente(X).view(-1)

print("\nDiferença média entre predições do w,b e do nn.Linear:",
      torch.mean(torch.abs(original_scores - linear_scores)).item())


Ao substituir os parâmetros w e b por uma camada nn.Linear e copiar os pesos manualmente, o modelo equivalente produziu exatamente as mesmas saídas. A diferença média entre as predições foi 0.0, demonstrando que nn.Linear implementa exatamente a mesma operação afim Xw + b utilizada na definição manual da SVM.